# Real or Rendered? Baseline (UnivFD)

Runs on a Colab GPU runtime. Cached features and results are written to `MyDrive/aml_real_or_rendered/`.

In [1]:
# Training size. 6 shards = Milestone 1 (10,505 images); 26 shards = about 44,000 images.
# Results and checkpoints for other sizes go to results_<n>shards/ and checkpoints_<n>shards/ on Drive.
%env ROR_TRAIN_SHARDS=6

env: ROR_TRAIN_SHARDS=6


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; df -h /content | tail -1; nproc; python --version

Tesla T4, 15360 MiB
overlay         113G   43G   71G  38% /
2
Python 3.13.15


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content
!test -d UniversalFakeDetect || git clone -q https://github.com/WisconsinAIVision/UniversalFakeDetect
!if test -d Real_or_Rendered; then git -C Real_or_Rendered pull -q; else git clone -q https://github.com/YassinZohair/Real_or_Rendered; fi
%cd /content/Real_or_Rendered
!pip install -q -r requirements.txt
!git log --oneline | head -3

/content
/content/Real_or_Rendered
208dd21 u1: LoRA on CLIP q/v attention, trained end to end with the head
c83f9c2 device-agnostic code,predict.py added for single demo runs
e7392a9 u2v2: platform downsizing before compression


## 1. Authors' code, weights and test data, unmodified

In [ ]:
!bash scripts/authors_eval.sh

Downloading...
From (original): https://drive.google.com/uc?id=1FXlGIRh_Ud3cScMgSVDbEWmPDmjcrm1t
From (redirected): https://drive.google.com/uc?id=1FXlGIRh_Ud3cScMgSVDbEWmPDmjcrm1t&confirm=t&uuid=a270869f-ddf8-464d-9b82-b373605b7087
To: /content/UniversalFakeDetect/diffusion_datasets.zip
100% 918M/918M [00:10<00:00, 86.6MB/s]
/content/UniversalFakeDetect/models/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
/content/UniversalFakeDetect/validate.py:21: DeprecationWarning: Please import `gaussian_filter` from the `scipy.ndimage` namespace; the `scipy.ndimage.filters` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.ndimage.filters import gaussian_filter
100%|███████████████████████████████████████| 890M/890M [00:09<00:0

## 2. GenImage: download, cache CLIP features

In [5]:
!python -m ror.data

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 62 files:   0% 0/62 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 0.00/1.80G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/3.83G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/3.83G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/3.83G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/3.83G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/3.83G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/3.83G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   1% 52.9M/3.83G [00:03<04:06, 15.3MB/s]
Reconstructing (incomplete total...):   2% 58.1M/3.83G [00:05<06:44, 9.33MB/s, 15.3MB/s  ]
Reconstructing (incomplete total...):   2% 65.5M/3.83G [00:05<05:52

In [6]:
!python -m ror.features baseline

/content/UniversalFakeDetect/models/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
100%|███████████████████████████████████████| 890M/890M [00:22<00:00, 41.1MiB/s]
train_stable_diffusion_v_1_4.npz: cached, skipping
test_ADM.npz: cached, skipping
test_BigGAN.npz: cached, skipping
test_glide.npz: cached, skipping
test_Midjourney.npz: cached, skipping
test_stable_diffusion_v_1_4.npz: cached, skipping
test_stable_diffusion_v_1_5.npz: cached, skipping
test_VQDM.npz: cached, skipping
test_wukong.npz: cached, skipping


## 3. Check the cached pipeline against the repo's model

In [ ]:
!python -m ror.check_faithful

## 4. Baseline results per generator

In [ ]:
!python -m ror.baseline

In [ ]:
!python -m ror.plots baseline
from IPython.display import Image, display
from ror.plots import FIGURES
for path in sorted(FIGURES.glob('baseline_*.png')):
    display(Image(str(path), width=900))

## 5. U2: equal file format + real-world damage

Needs the GenImage files from step 2 (`python -m ror.data`) on this runtime.

In [ ]:
!python -m ror.features u2

In [ ]:
!python -m ror.u2

In [ ]:
!python -m ror.plots u2
from IPython.display import Image, display
from ror.plots import FIGURES
for path in sorted(FIGURES.glob('u2_*.png')):
    display(Image(str(path), width=900))

## 6. U1: LoRA on CLIP's attention

Low-rank adapters on the query and value projections of all 24 CLIP layers, trained end to end with the head.
Needs the GenImage files from step 2 and the baseline heads from step 4, at the same `ROR_TRAIN_SHARDS`.
If the session drops, re-run the same cell: training resumes from its last validation, and finished runs are not retrained.

In [7]:
# Zero-init check: the adapted model must reproduce the cached baseline features, and only the adapters get gradients.
!python -m ror.u1 check

/content/UniversalFakeDetect/models/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
zero-init: max |feature difference| 0.00781, max |probability difference| 0.00015, same decision 100.0%
rank 8: 786,432 adapter parameters = 0.26% of CLIP's vision encoder (304,752,640); trainable tensors: 98 (adapters + head)
gradients: B matrices with a gradient in 24/24 layers, frozen tensors with a gradient: 0


In [8]:
# Rank sweep: one seed, clean test images only.
for rank in [4, 8, 16]:
    !python -m ror.u1 train --rank {rank} --seed 0

/content/UniversalFakeDetect/models/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
step 0 (= baseline): val acc 0.9305
u1_r4 seed 0: 9455 images, 295 steps of 32, validating every 73
epoch 1:  24% 72/295 [01:35<05:12,  1.40s/it]step 73: train loss 0.2310, val acc 0.9486
epoch 1:  49% 145/295 [03:34<03:31,  1.41s/it]step 146: train loss 0.1603, val acc 0.9695
epoch 1:  74% 218/295 [05:31<01:47,  1.40s/it]step 219: train loss 0.1274, val acc 0.9752
epoch 1:  99% 291/295 [07:29<00:05,  1.40s/it]step 292: train loss 0.1232, val acc 0.9781
epoch 1: 100% 294/295 [07:48<00:03,  3.60s/it]step 295: train loss 0.1951, val acc 0.9781
epoch 1: 100% 295/295 [08:05<00:00,  1.65s/it]
kept step 292 of 295 (val acc 0.9781)
test_ADM_u1_r4_seed0.npz: 4000 imag

In [9]:
!python -m ror.u1 report

                                                 acc  real_acc  fake_acc    ap   auc
protocol model          generator                                                   
resize   u1_r16         ADM                     51.7      97.1       6.4  58.0  57.7
                        BigGAN                  58.0      98.2      17.8  82.0  84.3
                        glide                   60.5      97.8      23.2  82.9  83.5
                        Midjourney              82.2      97.6      66.9  95.5  95.3
                        stable_diffusion_v_1_4  98.6      98.2      99.0  99.9  99.9
                        stable_diffusion_v_1_5  98.1      97.5      98.6  99.8  99.8
                        VQDM                    52.8      98.6       6.9  71.3  73.5
                        wukong                  95.2      98.2      92.2  99.3  99.2
         u1_r4          ADM                     51.5      95.9       7.2  56.0  56.0
                        BigGAN                  57.5      96.6   

In [10]:
# Best rank from the sweep, 3 seeds, also tested on the damaged copies (seed 0 is not retrained).
BEST_RANK = 8
for seed in [0, 1, 2]:
    !python -m ror.u1 train --rank {BEST_RANK} --seed {seed} --damaged

/content/UniversalFakeDetect/models/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
test_ADM_u1_r8_seed0.npz: cached, skipping
test_BigGAN_u1_r8_seed0.npz: cached, skipping
test_glide_u1_r8_seed0.npz: cached, skipping
test_Midjourney_u1_r8_seed0.npz: cached, skipping
test_stable_diffusion_v_1_4_u1_r8_seed0.npz: cached, skipping
test_stable_diffusion_v_1_5_u1_r8_seed0.npz: cached, skipping
test_VQDM_u1_r8_seed0.npz: cached, skipping
test_wukong_u1_r8_seed0.npz: cached, skipping
                                        acc  real_acc  fake_acc    ap   auc
protocol model generator                                                   
resize   u1_r8 ADM                     51.9      96.9       7.0  57.8  57.2
               BigGAN                  57.

In [ ]:
# B+U1+U2: the same adapters trained on U2 v2 damage with equal file format, starting from the U2 v2 head.
BEST_RANK = 8  # chosen by the rank sweep
for seed in [0, 1, 2]:
    !python -m ror.u1 train --rank {BEST_RANK} --seed {seed} --u2 --damaged

## 7. U3: noise branch fused with CLIP

Fixed SRM and constrained high-pass filters, then a small CNN trained from scratch on 128 px crops at the image's own resolution.
CLIP stays frozen. The noise head starts at zero, so B+U3 starts exactly as B. Two fusions: concat and a learned gate.
Needs the GenImage files (step 2). The crops are cached on this runtime's disk (several GB), so rebuild them in a new session.

In [ ]:
!python -m ror.patches

In [ ]:
for fusion in ['concat', 'gate']:
    for seed in [0, 1, 2]:
        !python -m ror.u3 train --fusion {fusion} --seed {seed}

## 8. Full model: U1 + U2 + U3

U3 trained on top of each seed's B+U1+U2 model from step 6 (its adapted CLIP is frozen at this stage).

In [ ]:
!python -m ror.patches full

In [ ]:
for fusion in ['concat', 'gate']:
    for seed in [0, 1, 2]:
        !python -m ror.u3 train --fusion {fusion} --seed {seed} --full

In [ ]:
# Every model, mean over the 8 generators, clean and damaged.
!python -m ror.u3 report

### Statistics and figures

Every model against B on the same test images: McNemar tests and bootstrap 95% CIs, plus the threshold diagnostic.
Then the Milestone 2 figures (`results/figures/m2_*.png`).

In [ ]:
!python -m ror.analysis

In [ ]:
!python -m ror.plots m2
from IPython.display import Image, display
from ror.plots import FIGURES
for path in sorted(FIGURES.glob('m2_*.png')):
    display(Image(str(path), width=900))

### Or: every Milestone 2 run in one cell

The B+U1+U2, U3 and full runs in order. Each step skips finished work, so after a disconnect run it again.

In [ ]:
!bash scripts/m2_runs.sh

## 9. Score your own images

The whole pipeline on single images, no cache: test view, frozen CLIP, a trained head, P(AI).
Needs only cells 2 and 3 (Drive for the head, the repos for the code).

In [ ]:
from google.colab import files
uploaded = files.upload()
names = ' '.join(f'"{n}"' for n in uploaded)
# Any head in checkpoints/ works, e.g. u2v2_seed0.pth for the U2 model, or a U1 checkpoint such as u1_r8_seed0.pt.
!python -m ror.predict {names} --head /content/drive/MyDrive/aml_real_or_rendered/checkpoints/retrained_sd14_seed0.pth